In [7]:
import yfinance as yf
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def prepare_data(ticker: str, start: str = '2016-01-01', seq_len: int = 30, split_frac: float = 0.8):
    df = yf.download(ticker, start=start, auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df['log_ret'] = np.log(df['Close'] / df['Close'].shift(1))
    df = df.dropna()

    split = int(split_frac * len(df))

    scaler = StandardScaler().fit(df['log_ret'].iloc[:split].values.reshape(-1, 1))
    scaled = scaler.transform(df['log_ret'].values.reshape(-1, 1)).flatten()

    def make_windows(s, seq):
        X = np.stack([s[i:i + seq] for i in range(len(s) - seq)])
        y = np.array([s[i + seq] for i in range(len(s) - seq)])
        return X, y

    Xtr, ytr = make_windows(scaled[:split], seq_len)
    Xte, yte = make_windows(scaled[split:], seq_len)

    to_tensor = lambda a: torch.tensor(a, dtype=torch.float32).unsqueeze(-1).to(device)
    Xtr, ytr, Xte, yte = to_tensor(Xtr), to_tensor(ytr), to_tensor(Xte), to_tensor(yte)

    return Xtr, ytr, Xte, yte, scaler, df

class PredictionModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers, output_dim):
        super().__init__()
        self.hidden_dim, self.num_layers = hidden_dim, num_layers
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_dim, device=x.device)
        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out[:, -1, :])


def train_model(Xtr, ytr, epochs=200, hidden_dim=32, num_layers=2, lr=1e-3, print_every=25):
    model = PredictionModel(1, hidden_dim, num_layers, 1).to(device)
    criterion = nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        pred = model(Xtr)
        loss = criterion(pred, ytr)
        opt.zero_grad()
        loss.backward()
        opt.step()
        if epoch % print_every == 0:
            print(f"  epoch {epoch:4d}  loss {loss.item():.6f}")

    return model


def evaluate(model, Xte, yte, scaler) -> dict:
    model.eval()
    with torch.no_grad():
        pred_scaled = model(Xte).cpu().numpy()
    true_scaled = yte.cpu().numpy()

    pred = scaler.inverse_transform(pred_scaled).flatten()
    true = scaler.inverse_transform(true_scaled).flatten()
    naive = np.zeros_like(true) 

    results = {}

    model_rmse = root_mean_squared_error(true, pred)
    naive_rmse = root_mean_squared_error(true, naive)
    results['rmse'] = {
        'model': model_rmse, 'baseline': naive_rmse,
        'beats_baseline': model_rmse < naive_rmse,
        'relative_improvement_pct': (naive_rmse - model_rmse) / naive_rmse * 100,
    }

 
    model_mae = mean_absolute_error(true, pred)
    naive_mae = mean_absolute_error(true, naive)
    results['mae'] = {
        'model': model_mae, 'baseline': naive_mae,
        'beats_baseline': model_mae < naive_mae,
        'relative_improvement_pct': (naive_mae - model_mae) / naive_mae * 100,
    }

    model_r2 = r2_score(true, pred)
    results['r2'] = {
        'model': model_r2, 'baseline': 0.0,
        'beats_baseline': model_r2 > 0.0,
    }

    true_dir = np.sign(true)
    pred_dir = np.sign(pred)
    model_dir_acc = (pred_dir == true_dir).mean()

    up_rate = (true_dir > 0).mean()
    majority_baseline = max(up_rate, 1 - up_rate)
    results['directional_accuracy'] = {
        'model': model_dir_acc, 'baseline': majority_baseline,
        'beats_baseline': model_dir_acc > majority_baseline,
        'up_days_pct_in_test': up_rate,
    }

    results['prediction_variance_ratio'] = pred.var() / true.var() if true.var() > 0 else float('nan')
    results['_pred'] = pred
    results['_true'] = true
    return results


def print_report(results: dict):
    print(f"\n{'='*70}")
    print(f"{'Metric':<24}{'Model':>12}{'Baseline':>12}{'Beats?':>10}")
    print(f"{'-'*70}")
    for name in ['rmse', 'mae', 'r2', 'directional_accuracy']:
        r = results[name]
        beats = "YES" if r['beats_baseline'] else "NO"
        print(f"{name:<24}{r['model']:>12.5f}{r['baseline']:>12.5f}{beats:>10}")
    print(f"{'='*70}")

    pvr = results['prediction_variance_ratio']
    print(f"\nPrediction variance / actual variance: {pvr:.3f}")
    if pvr < 0.1:
        print(
            "The model is likely just predicting values close to the mean rather than capturing " 
            "real movement. " \
            "Treat any RMSE/MAE win alongside this number, not in isolation." 
        )

    n_wins = sum(results[m]['beats_baseline'] for m in ['rmse', 'mae', 'r2', 'directional_accuracy'])
    print(f"\nBeats baseline on {n_wins}/4 metrics.")
    if n_wins <= 1:
        print(
            "Honest read: little to no evidence of real predictive skill at "
            "this horizon. Worth reporting as-is rather than tuning until "
            "the numbers look better -- that risks overfitting to this one "
            "test split rather than finding real signal."
        )

if __name__ == "__main__":

    TICKER = "NTES"
    print(f"Preparing data for {TICKER}")
    Xtr, ytr, Xte, yte, scaler, df = prepare_data(TICKER)

    print(f"Train windows: {len(Xtr)}, Test windows: {len(Xte)}")
    print("Training LSTM")
    model = train_model(Xtr, ytr, epochs=200)

    print("\nEvaluating model")
    results = evaluate(model, Xte, yte, scaler)
    print_report(results)
    pred = results['_pred']
    true = results['_true']

    deciles = pd.qcut(pred, 10, labels=False, duplicates='drop')
    up_rate_by_decile = pd.Series((true > 0).astype(float)).groupby(deciles).mean()
    print("\nUp-rate by prediction decile (0=lowest predicted, N=highest):")
    print(up_rate_by_decile)

    mean_ret_by_decile = pd.Series(true).groupby(deciles).mean()
    print("\nMean realized return by decile:")
    print(mean_ret_by_decile)

Preparing data for NTES
Train windows: 2124, Test windows: 509
Training LSTM
  epoch    0  loss 1.026225
  epoch   25  loss 0.993016
  epoch   50  loss 0.989793
  epoch   75  loss 0.987511
  epoch  100  loss 0.986953
  epoch  125  loss 0.986254
  epoch  150  loss 0.984566
  epoch  175  loss 0.976828

Evaluating model

Metric                         Model    Baseline    Beats?
----------------------------------------------------------------------
rmse                         0.02494     0.02296        NO
mae                          0.01757     0.01624        NO
r2                          -0.18156     0.00000        NO
directional_accuracy         0.50884     0.51277        NO

Prediction variance / actual variance: 0.156

Beats baseline on 0/4 metrics.
Honest read: little to no evidence of real predictive skill at this horizon. Worth reporting as-is rather than tuning until the numbers look better -- that risks overfitting to this one test split rather than finding real signal.

Up-ra